## 7.1 — Snowflake Cleanup

Drops all tables and views in the configured Snowflake schema.

**Preserved (NOT touched):**
- Storage integration (`TPCDS_S3_INTEGRATION`)
- External stage (`TPCDS_EXTERNAL_STAGE`)
- File format (`TPCDS_CSV_FORMAT`)
- Database, schema, warehouse themselves
- Raw `.dat` files in S3, logs, results, JMeter test files, streams

**Dropped:**
- Every table / view under `{sf_database}.{sf_schema}`


In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
%pip install "snowflake-connector-python>=3.11" cryptography --quiet
dbutils.library.restartPython()

In [ ]:
# --- Create Widgets ---
dbutils.widgets.removeAll()
dbutils.widgets.text("sf_account_locator", "", "Snowflake Account Locator")
dbutils.widgets.text("sf_region", "", "Snowflake Region")
dbutils.widgets.text("sf_database", "TPCDS", "Snowflake Database")
dbutils.widgets.text("sf_schema", "BENCHMARK", "Snowflake Schema")
dbutils.widgets.text("sf_role", "SYSADMIN", "Snowflake Role")
dbutils.widgets.text("sf_user", "", "Snowflake User")
dbutils.widgets.text("sf_warehouse", "", "Snowflake Warehouse (optional; cleanup only needs a session)")
dbutils.widgets.text("dbx_catalog", "tpcds_sf_catalog", "Databricks Catalog (for RSA key volume)")
dbutils.widgets.text("dbx_schema", "tpcds", "Databricks Schema (for RSA key volume)")
dbutils.widgets.text("dbx_volume", "tpcds_raw_data", "Databricks Volume (for RSA key)")
dbutils.widgets.text("table_prefix", "", "Table Prefix filter (optional; blank = drop ALL)")
dbutils.widgets.dropdown("dry_run", "false", ["true", "false"], "Dry Run")

In [ ]:
import os

params = dbutils.widgets.getAll()

sf_account = params["sf_account_locator"]
sf_region = params["sf_region"]
sf_database = params["sf_database"]
sf_schema = params["sf_schema"]
sf_role = params["sf_role"]
sf_user = params["sf_user"]
sf_warehouse = params.get("sf_warehouse", "").strip() or None
prefix = params.get("table_prefix", "").strip()
dry_run = params.get("dry_run", "false").lower() == "true"

volume_base = f"/Volumes/{params['dbx_catalog']}/{params['dbx_schema']}/{params['dbx_volume']}"
auth_key_path = os.path.join(volume_base, "test_files", "rsa_key.p8")

if not sf_account or not sf_user:
    raise RuntimeError("Validation Error: sf_account_locator and sf_user are required.")
if not os.path.exists(auth_key_path):
    raise RuntimeError(
        f"Validation Error: RSA key not found at {auth_key_path}. "
        f"Run '4.1 - Snowflake - JMX Generation' first."
    )

print(f"Target: {sf_database}.{sf_schema}")
print(f"Prefix filter: {prefix!r}" if prefix else "Prefix filter: (none — every table will be dropped)")
print(f"Dry run: {dry_run}")

In [ ]:
import snowflake.connector
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(auth_key_path, "rb") as f:
    p_key = serialization.load_pem_private_key(f.read(), password=None, backend=default_backend())
pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
account = sf_account if "-" in sf_account else f"{sf_account}.{sf_region}"

conn = snowflake.connector.connect(
    user=sf_user, account=account, private_key=pkb,
    role=sf_role, database=sf_database, schema=sf_schema, warehouse=sf_warehouse,
)

targets = []  # list of (name, kind)
try:
    cur = conn.cursor()
    # SHOW TABLES returns BASE TABLES, EXTERNAL TABLES, ICEBERG TABLES, DYNAMIC TABLES
    cur.execute(f"SHOW TABLES IN SCHEMA {sf_database}.{sf_schema}")
    cols = [d[0].lower() for d in cur.description]
    for row in cur.fetchall():
        d = dict(zip(cols, row))
        name = d["name"]
        if prefix and not name.startswith(prefix):
            continue
        # kind column indicates the kind; default to TABLE
        kind_str = (d.get("kind") or "TABLE").upper()
        if "ICEBERG" in kind_str:
            targets.append((name, "ICEBERG TABLE"))
        elif "DYNAMIC" in kind_str:
            targets.append((name, "DYNAMIC TABLE"))
        elif "EXTERNAL" in kind_str:
            targets.append((name, "EXTERNAL TABLE"))
        else:
            targets.append((name, "TABLE"))

    cur.execute(f"SHOW VIEWS IN SCHEMA {sf_database}.{sf_schema}")
    cols = [d[0].lower() for d in cur.description]
    for row in cur.fetchall():
        d = dict(zip(cols, row))
        name = d["name"]
        if prefix and not name.startswith(prefix):
            continue
        kind_str = (d.get("kind") or "VIEW").upper()
        if "MATERIALIZED" in kind_str:
            targets.append((name, "MATERIALIZED VIEW"))
        else:
            targets.append((name, "VIEW"))

    # Drop views first, tables second
    targets.sort(key=lambda t: (("VIEW" not in t[1]), t[0]))

    print(f"Found {len(targets)} object(s) to drop")
    for name, kind in targets:
        print(f"  [{kind}] {sf_database}.{sf_schema}.{name}")
finally:
    conn.close()

In [ ]:
# Execute drops
import snowflake.connector

conn = snowflake.connector.connect(
    user=sf_user, account=account, private_key=pkb,
    role=sf_role, database=sf_database, schema=sf_schema, warehouse=sf_warehouse,
)

dropped = 0
failed = []
try:
    cur = conn.cursor()
    for name, kind in targets:
        stmt = f'DROP {kind} IF EXISTS "{sf_database}"."{sf_schema}"."{name}"'
        if dry_run:
            print(f"DRY RUN -> {stmt}")
            continue
        try:
            cur.execute(stmt)
            dropped += 1
            print(f"dropped [{kind}] {name}")
        except Exception as e:
            failed.append((name, kind, str(e)))
            print(f"FAILED [{kind}] {name}: {e}")
finally:
    conn.close()

print(f"\nSummary: {dropped}/{len(targets)} dropped, {len(failed)} failed")
if failed:
    for n, k, err in failed:
        print(f"  - {k} {n}: {err[:200]}")
    raise RuntimeError(f"Cleanup completed with {len(failed)} failure(s)")

In [ ]:
# Verify: stage/file format/integration/warehouse untouched
conn = snowflake.connector.connect(
    user=sf_user, account=account, private_key=pkb,
    role=sf_role, database=sf_database, schema=sf_schema, warehouse=sf_warehouse,
)
try:
    cur = conn.cursor()
    for show in [
        f"SHOW STAGES IN SCHEMA {sf_database}.{sf_schema}",
        f"SHOW FILE FORMATS IN SCHEMA {sf_database}.{sf_schema}",
        f"SHOW INTEGRATIONS LIKE 'TPCDS_%'",
        f"SHOW WAREHOUSES LIKE '{sf_warehouse}'",
    ]:
        cur.execute(show)
        cols = [d[0].lower() for d in cur.description]
        rows = cur.fetchall()
        print(f"{show}: {len(rows)} row(s)")
        for r in rows:
            d = dict(zip(cols, r))
            print(f"  - {d.get('name')}")
finally:
    conn.close()